# 01 Network and differentiable solver

The plant in every experiment is the IEEE 33-bus radial feeder (Baran and Wu, 1989, as `case33bw` in pandapower), solved
with branch-flow DistFlow by a fixed number of backward/forward sweeps written in PyTorch, so that gradients pass through
the solver. Bus 0 is the substation (slack), all quantities are per unit, and a discharging battery (P > 0) reduces the
net consumption at its bus.

This page shows the check of the solver against pandapower's AC power flow. The pre-set target was a voltage error below
0.001 pu.

```{include} ../_caveats.md
```

In [ ]:
from IPython.display import Markdown, display
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from common import *
m1 = load("m1_validation.json")
display(Markdown(provenance(m1, "m1_validation.json")))
cfg = m1["config"]
display(Markdown(f"Random load cases: **{cfg['n_random']}**, seed {cfg['seed']}, sweeps per solve: **{cfg['n_iter']}**, "
                 f"target: **{cfg['target_pu']} pu**. Versions: " + ", ".join(f"{k} {v}" for k, v in m1['versions'].items()) + "."))

## Error against pandapower

In [ ]:
s = m1["summary"]
labels = {"max_voltage_error_pu_float32": "max voltage error, float32 (pu)",
          "max_voltage_error_pu_float64": "max voltage error, float64 (pu)",
          "mean_voltage_error_pu_float32": "mean voltage error, float32 (pu)",
          "max_loss_error_pu_float32": "max loss error, float32 (pu)",
          "max_slack_p_error_pu_float32": "max slack active power error, float32 (pu)",
          "max_slack_q_error_pu_float32": "max slack reactive power error, float32 (pu)",
          "passes_target": f"below the {cfg['target_pu']} pu target"}
table([{"quantity": labels.get(k, k.replace("_", " ")), "value": fmt(v)} for k, v in s.items()], index="quantity")

## Named operating points

Voltages and losses from the differentiable solver, and its largest voltage difference to pandapower.

In [ ]:
rows = []
for name, r in m1["named_scenarios"].items():
    rows.append({"case": name, "V min (pu)": fmt(r["v_min"], 4), "at bus": r["v_min_bus"], "V max (pu)": fmt(r["v_max"], 4),
                 "losses (kW)": fmt(r["loss_kw"], 1), "max |ΔV| float32": fmt(r["max_dv_float32"], 3),
                 "max |ΔV| float64": fmt(r["max_dv_float64"], 3)})
table(rows, index="case")

## How many sweeps are needed

Training uses 8 sweeps per solve (cheaper backpropagation) and evaluation uses 15.

In [ ]:
sweep = {int(k): v for k, v in m1["n_iter_sweep"].items()}
fig, ax = plt.subplots(figsize=(6.4, 3.4))
ax.semilogy(list(sweep), list(sweep.values()), marker="o", markersize=5, color=SERIES[0])
ax.axhline(cfg["target_pu"], color=INK_2, linewidth=1, linestyle="--")
ax.text(max(sweep), cfg["target_pu"] * 1.4, "target", ha="right", color=INK_2)
for n, label in [(8, "training"), (15, "evaluation")]:
    ax.annotate(label, (n, sweep[n]), textcoords="offset points", xytext=(4, 8), color=INK_2)
ax.set_xlabel("backward/forward sweeps")
ax.set_ylabel("max voltage error vs pandapower (pu)")
ax.set_title("Solver error against the number of sweeps (float64)", loc="left")
plt.show()

In [ ]:
low = m1["nominal_buses_below_0.95"]
display(Markdown(f"Without control at the nominal case33bw load, **{len(low)}** buses are below 0.95 pu. This is why the "
                 "base scenario scales the load by 0.7 (chosen before any time series or controller result was seen)."))